# HypRAG Benchmark — Google Colab (T4 GPU)
Runs the full hybrid benchmark (FAISS / HypRAG / BM25+RRF) on the CPython stdlib corpus.

> **Runtime → Change runtime type → T4 GPU** before running.

## Cell 1 — GPU check

In [ ]:
import subprocess, torch

result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No nvidia-smi found')

print(f'torch {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
else:
    print('WARNING: no GPU — encoding will be slow. Set runtime to T4.')

## Cell 2 — Clone HypRAG from GitHub
Always clones fresh so you get the latest `master`. Re-running this cell in the same session just skips if the folder already exists.

In [ ]:
import os

REPO = 'https://github.com/wetzy/hyprag.git'

if not os.path.isdir('hyprag'):
    !git clone {REPO}
else:
    print('Repo already cloned — pulling latest master...')
    !git -C hyprag pull origin master

%cd hyprag
!git log --oneline -5

## Cell 3 — Install dependencies
Auto-detects CUDA version and installs `faiss-gpu-cu12`, `faiss-gpu-cu11`, or `faiss-cpu` as fallback. For 16k vectors, FAISS search is fast on CPU — the GPU win is in sentence-transformers encoding, which uses CUDA automatically via PyTorch.

In [ ]:
import subprocess, sys

# Detect CUDA version to pick the right faiss wheel
# faiss-gpu was removed from PyPI; new packages are faiss-gpu-cu11 / faiss-gpu-cu12
def _cuda_major():
    try:
        out = subprocess.check_output(['nvcc', '--version'], stderr=subprocess.DEVNULL).decode()
        for part in out.split():
            if part.startswith('V'):
                return int(part[1:].split('.')[0])
    except Exception:
        pass
    try:
        import torch
        if torch.cuda.is_available():
            return int(torch.version.cuda.split('.')[0])
    except Exception:
        pass
    return None

cuda = _cuda_major()
if cuda is None:
    faiss_pkg = 'faiss-cpu'
elif cuda >= 12:
    faiss_pkg = 'faiss-gpu-cu12'
else:
    faiss_pkg = 'faiss-gpu-cu11'

print(f'CUDA major: {cuda}  →  installing {faiss_pkg}')
ret = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', faiss_pkg], capture_output=True)
if ret.returncode != 0:
    print(f'  {faiss_pkg} not available, falling back to faiss-cpu')
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'faiss-cpu'], check=True)

# Core deps
!pip install -q geoopt sentence-transformers psutil anthropic

# Install hyprag without re-resolving faiss (avoids downgrade to faiss-cpu)
!pip install -q --no-deps -e .

# Smoke-test
import hyprag; print('hyprag import OK')
import faiss; print(f'faiss {faiss.__version__}')
import torch; print(f'torch {torch.__version__}, CUDA {torch.cuda.is_available()}')

## Cell 4 — Sparse-clone CPython `Lib/` only
Downloads only the stdlib source (~50 MB) instead of the full CPython repo (~1 GB).

In [ ]:
import os

if not os.path.isdir('cpython/Lib'):
    !git clone --depth 1 --filter=blob:none --sparse \
        https://github.com/python/cpython.git cpython
    !git -C cpython sparse-checkout set Lib
else:
    print('cpython/Lib already present, skipping clone')

py_count = !find cpython/Lib -name '*.py' | wc -l
print(f'.py files in cpython/Lib: {py_count[0].strip()}')

## Cell 5 — Structural benchmark (fast, no model download)
Validates chunking throughput, index build time, memory, and search latency using random vectors. Takes ~30 s.

In [ ]:
!python -m benchmarks.run_benchmark \
    --cpython-lib cpython/Lib \
    --structural-only

import json
s = json.load(open('benchmarks/results/structural.json'))
print(f"\nCorpus  : {s['corpus_files']} files, {s['corpus_chunks']:,} chunks")
print(f"Chunking: {s['chunks_per_sec']:,.0f} chunks/s")
print(f"\nLatency (ms/query, k=10):")
print(f"  FAISS  : {s['faiss_search_ms_per_query']}")
print(f"  HypRAG : {s['hyprag_search_ms_per_query']}")

## Cell 6 — Full semantic benchmark (BGE encoder, GPU)
Downloads `BAAI/bge-base-en-v1.5` (~440 MB first run, cached after), encodes 16k chunks on GPU, then runs all 20 labeled queries through FAISS / HypRAG / Hybrid. Takes ~5 min on T4.

In [ ]:
!python -m benchmarks.run_benchmark \
    --cpython-lib cpython/Lib \
    --encoder BAAI/bge-base-en-v1.5

print('\nDone. Results in benchmarks/results/')

## Cell 7 — Results summary

In [ ]:
import json

r = json.load(open('benchmarks/results/semantic.json'))

print(f"Encoder : {r['encoder_model']}")
print(f"Queries : {r['n_queries']}")
print()
print(f"{'Retriever':<25} {'Recall@5':>10} {'Precision@5':>12}")
print('-' * 50)
rows = [
    ('FAISS',            r['recall_at_k_faiss'],            r['precision_at_k_faiss']),
    ('HypRAG (k-NN)',    r['recall_at_k_hyprag'],           r['precision_at_k_hyprag']),
    ('HypRAG + expand',  r['recall_at_k_hyprag_expanded'],  r['precision_at_k_hyprag_expanded']),
    ('Hybrid (RRF)',     r['recall_at_k_hybrid'],           r['precision_at_k_hybrid']),
    ('Hybrid + expand',  r['recall_at_k_hybrid_expanded'],  r['precision_at_k_hybrid_expanded']),
]
for name, rec, prec in rows:
    print(f'{name:<25} {rec:>10.3f} {prec:>12.3f}')

# Highlight lift vs FAISS baseline
baseline = r['recall_at_k_faiss']
best = r['recall_at_k_hybrid_expanded']
print(f"\nHybrid+expand lift vs FAISS: {(best - baseline) / baseline * 100:+.1f}%")

## Cell 8 — Per-query breakdown

In [ ]:
import json
r = json.load(open('benchmarks/results/semantic.json'))

print(f"{'Query':<45} {'FAISS':>6} {'Hyp+exp':>8} {'Hyb+exp':>8}")
print('-' * 72)
for q in r['per_query']:
    print(
        f"{q['query'][:44]:<45} "
        f"{q['faiss']['recall']:>6.2f} "
        f"{q['hyprag_expanded']['recall']:>8.2f} "
        f"{q['hybrid_expanded']['recall']:>8.2f}"
    )

# Zero-recall queries still failing (summary candidates)
zeros = [q['query'] for q in r['per_query'] if q['hybrid_expanded']['recall'] == 0.0]
print(f"\nStill zero-recall ({len(zeros)}): {zeros}")

## Cell 9 — (Optional) Generate LLM summaries for zero-recall chunks
Costs ~$0.65 for all 16k chunks with Haiku. Set `ANTHROPIC_API_KEY` first.
Skip this cell if you want to evaluate hybrid-only first.

In [ ]:
import os
os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'  # paste your key here

!python -m benchmarks.generate_summaries \
    --cpython-lib cpython/Lib \
    --out benchmarks/results/summaries.json \
    --model claude-haiku-4-5-20251001 \
    --concurrency 50

import json
cache = json.load(open('benchmarks/results/summaries.json'))
print(f"Summaries generated: {cache['n_summaries']:,}")

## Cell 10 — (Optional) Re-run benchmark with summaries
Compares hybrid+expand with vs without LLM summaries. Run Cell 9 first.

In [ ]:
!python -m benchmarks.run_benchmark \
    --cpython-lib cpython/Lib \
    --encoder BAAI/bge-base-en-v1.5 \
    --summaries benchmarks/results/summaries.json

import json
r = json.load(open('benchmarks/results/semantic.json'))
print(f"Hybrid + expand + summaries  Recall@5: {r['recall_at_k_hybrid_expanded']:.3f}")

## Cell 11 — Push results back to GitHub
Commits `benchmarks/results/` so the numbers survive after the Colab session ends.
Requires a GitHub personal access token with `repo` scope.

In [ ]:
import os
GH_TOKEN = 'ghp_...'  # paste a GitHub PAT with repo scope

!git config user.email 'wetzy@users.noreply.github.com'
!git config user.name  'wetzy'

remote_auth = f'https://{GH_TOKEN}@github.com/wetzy/hyprag.git'
!git remote set-url origin {remote_auth}

!git add benchmarks/results/
!git commit -m 'chore: add BGE hybrid benchmark results from Colab T4' || echo 'nothing to commit'
!git push origin master

print('Done — results pushed to master.')